# Structured Output with `with_structured_output`

This notebook demonstrates a modern approach to structuring LLM outputs using the `.with_structured_output()` method. This method simplifies the process of getting structured data (like JSON) from an LLM without complex manual parsing.

### Feature Comparison: Pydantic Parsing vs `with_structured_output`

| FEATURE | PYDANTIC PARSING | WITH_STRUCTURED_OUTPUT |
| :--- | :---: | :---: |
| **Ease of Use** | ❌ Hard<br>(Manual prompts) | ✅ Easy<br>(One method call) |
| **Reliability** | ⚠️ Medium<br>(Prompt dependent) | ✅ High<br>(Tool calling/JSON) |
| **Model Support** | ✅ Universal<br>(Any LLM works) | ⚠️ Limited<br>(Needs tool calling) |
| **Prompt Control** | ✅ Complete<br>(Full customization) | ❌ Limited<br>(Automatic handling) |

## 1. Imports and Environment Setup

In [1]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.react.agent import create_react_agent
from langchain_tavily import TavilySearch
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from prompts import main_prompt
from schema import AgentResponse
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

## 2. Initialize Tools

In [2]:
# Initialize Tavily search tool
tavily_search = TavilySearch(max_results=5, topic="general")
tools = [tavily_search]

print(f"Tools initialized: {[tool.name for tool in tools]}")

Tools initialized: ['tavily_search']


## 3. Initialize LLM

In [3]:
# Initialize Google Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("LLM initialized successfully")

LLM initialized successfully


## 4. Configure Structured Output

Here we use the `.with_structured_output()` method. This binds the `AgentResponse` Pydantic model to the LLM, instructing it to return output that matches this schema. This is often more reliable than asking for JSON in the prompt text.

In [4]:
# Configure LLM for structured output
structured_llm = llm.with_structured_output(AgentResponse)

print("LLM configured with structured output schema")

LLM configured with structured output schema


## 5. Setup Prompt and Agent

We use the same ReAct prompt but we **remove** the manual format instructions because `with_structured_output` handles the formatting automatically.

In [5]:
# Create prompt template with EMPTY format instructions
react_prompt_with_format_instructions = PromptTemplate(
    template=main_prompt, 
    input_variables=["input", "agent_scratchpad", "tool_names"]
).partial(format_instructions="")

print("Prompt template created")

Prompt template created


In [7]:
# Create ReAct agent
agent = create_react_agent(
    llm=llm, 
    tools=tools, 
    prompt=react_prompt_with_format_instructions
)

print("Agent created successfully")

Agent created successfully


In [8]:
# Create agent executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True
)

# Create a runnable to fetch the output string from the agent result
fetch_output = RunnableLambda(lambda x: x["output"])

print("Agent executor and fetcher created")

Agent executor and fetcher created


## 6. Create and Run Chain

The chain connects: `Agent Executor` -> `Output Fetcher` -> `Structured LLM`

In [9]:
# Define the chain
chain = agent_executor | fetch_output | structured_llm

# Execute the chain
result = chain.invoke(input={"input": "What are the latest news about AI in the world"})

print("\n" + "="*50)
print("FINAL STRUCTURED RESULT:")
print("="*50)
print(result)
print(f"\nType of result: {type(result)}")



> Entering new AgentExecutor chain...
Action: tavily_search
Action Input: latest news about AI in the world{'query': 'latest news about AI in the world', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.artificialintelligence-news.com/', 'title': 'AI News | Latest News | Insights Powering AI-Driven Business ...', 'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.', 'score': 0.7653308, 'raw_content': None}, {'url': 'https://www.reuters.com/technology/artificial-intelligence/', 'title': 'AI News | Latest Headlines and Developments', 'content': 'Explore the latest artificial intelligence news with Reuters - from AI breakthroughs and technology trends to regulation, ethics, business and global impact', 'score': 0.72395444, 'raw_content': None}, {'url': 'https://www.nbcnews.com/artificial-intelligence', 'title': 'Artificial intelligence', 'conten